In [1]:
import os
import math
import shutil 
import random
import itertools
import numpy as np
import pandas as pd 
from glob import glob

In [2]:
train_csv_file="fold1_train.csv" 
test_csv_file="fold1_test.csv" 

path="Datasets/1.HAM10000/"
data_root="rasm/siamese/fold1/"

In [3]:
output_filename_train_genuine = "ham10000_siamese_train_genuine_pairs.csv"
output_filename_train_imposter = "ham10000_siamese_train_imposter_pairs.csv"  
output_filename_test_genuine = "ham10000_siamese_test_genuine_pairs.csv"
output_filename_test_imposter = "ham10000_siamese_test_imposter_pairs.csv"  

# Train

In [4]:
df_train=pd.read_csv(data_root+train_csv_file, header=None) 
train_file_list=df_train[0].tolist()

In [5]:
train_folderlist=glob("rasm/siamese/fold1/train_aug/*/*")
random.shuffle(train_folderlist)

In [6]:
# Genuine Pairs
train_genuine_pairs=[]
for x in train_folderlist:
    files=glob(x+"/*")
    for y in files:
        path=y.split('/')[:-1]
        path="/".join(path)
        img1=path+"/"+y.split('/')[-2]+'_Original.png'
        img2=y
        train_genuine_pairs.append([img1, img2])

train_genuine_pairs = np.array(train_genuine_pairs)
np.random.shuffle(train_genuine_pairs)
print('train genuine pairs shape : ', train_genuine_pairs.shape)

train genuine pairs shape :  (36612, 2)


In [7]:
print(train_genuine_pairs[0][0])
print(train_genuine_pairs[0][1])

rasm/siamese/fold1/train_aug/nv/ISIC_0032487/ISIC_0032487_Original.png
rasm/siamese/fold1/train_aug/nv/ISIC_0032487/ISIC_0032487_RGBShift.png


In [8]:
same=0
for i in range(train_genuine_pairs.shape[0]):
    if train_genuine_pairs[i][0].split('_')[3] != train_genuine_pairs[i][1].split('_')[3]:
        same = same + 1

print(same)

0


In [9]:
num_train_genuine_pairs=train_genuine_pairs.shape[0]
print("num train genuine pairs : ", num_train_genuine_pairs)

num train genuine pairs :  36612


In [10]:
train_folderlist[0]

'rasm/siamese/fold1/train_aug/bcc/ISIC_0028688'

In [11]:
# Imposter pairs
train_imposter_pairs=[]
for x in train_folderlist:   
    current_folder_filenames=np.array(glob(x+"/*"))
    current_folder_filenames_repeated=np.tile(current_folder_filenames,54)  
    np.random.shuffle(current_folder_filenames_repeated)
    current_folder_filenames_repeated=current_folder_filenames_repeated[:54]

    tfl = np.array(train_folderlist)
    other_folderlist=[]
    c = x.split('/')[4]  
    for t in tfl:
        cl=t.split('/')[4]     
        if c != cl:
            other_folderlist.append(t)
    
    temp = []
    for z in other_folderlist:
        temp.append(glob(z+"/*"))

    temp=np.array(temp)
    temp=temp.reshape(-1)
    np.random.shuffle(temp)
    temp=temp[:54]
    other_folder_filenames=temp

    train_imposter_pairs.append(np.column_stack((current_folder_filenames_repeated, other_folder_filenames)))

train_imposter_pairs=np.array(train_imposter_pairs)
train_imposter_pairs = train_imposter_pairs.reshape(train_imposter_pairs.shape[0]*train_imposter_pairs.shape[1],train_imposter_pairs.shape[2])
np.random.shuffle(train_imposter_pairs)  

print('train imposter pairs shape : ', train_imposter_pairs.shape)

train imposter pairs shape :  (36612, 2)


In [12]:
print(train_imposter_pairs[0][0])
print(train_imposter_pairs[0][1])

rasm/siamese/fold1/train_aug/nv/ISIC_0024725/ISIC_0024725_RGBShift.png
rasm/siamese/fold1/train_aug/bkl/ISIC_0025703/ISIC_0025703_RandomRain.png


In [13]:
# check same file possibility
same=0
for i in range(train_imposter_pairs.shape[0]):
    if train_imposter_pairs[i][0].split("_")[3] == train_imposter_pairs[i][1].split("_")[3]:
        same=same+1

print('same file duplicated in imposter pairs: ', same)

same file duplicated in imposter pairs:  0


In [14]:
# check same class possibility
same=0
for i in range(train_imposter_pairs.shape[0]):
    if train_imposter_pairs[i][0].split("/")[4] == train_imposter_pairs[i][1].split("/")[4]:
        same=same+1

print('same class duplicated in imposter pairs : ', same)

same class duplicated in imposter pairs :  0


In [15]:
# Train Labels and Save
labels_train_genuine=np.zeros(train_genuine_pairs.shape[0], dtype=np.int8)
labels_train_genuine=labels_train_genuine.reshape(-1,1)
train_genuine_pairs = np.concatenate((train_genuine_pairs, labels_train_genuine), axis=1)

labels_train_imposter=np.ones(train_imposter_pairs.shape[0], dtype=np.int8)
labels_train_imposter=labels_train_imposter.reshape(-1,1)
train_imposter_pairs = np.concatenate((train_imposter_pairs, labels_train_imposter), axis=1)

In [16]:
# save csv files
df=pd.DataFrame(train_genuine_pairs)
df.to_csv(data_root+output_filename_train_genuine, index=False, header=False)   
df=pd.DataFrame(train_imposter_pairs)
df.to_csv(data_root+output_filename_train_imposter, index=False, header=False)  

# Test

In [17]:
df_test=pd.read_csv(data_root+test_csv_file, header=None)
test_file_list=df_test[0].tolist()

In [18]:
test_folderlist=glob("rasm/siamese/fold1/test_aug/*/*")
random.shuffle(test_folderlist)

In [19]:
# Genuine Pairs
test_genuine_pairs=[]
for x in test_folderlist:
    files=glob(x+"/*")
    for y in files:
        path=y.split('/')[:-1]
        path="/".join(path)
        img1=path+"/"+y.split('/')[-2]+'_Original.png'
        img2=y
        test_genuine_pairs.append([img1, img2])

test_genuine_pairs = np.array(test_genuine_pairs)
np.random.shuffle(test_genuine_pairs)
print('train genuine pairs shape : ', test_genuine_pairs.shape)

train genuine pairs shape :  (8964, 2)


In [20]:
print(test_genuine_pairs[0][0])
print(test_genuine_pairs[0][1])

rasm/siamese/fold1/test_aug/bcc/ISIC_0034160/ISIC_0034160_Original.png
rasm/siamese/fold1/test_aug/bcc/ISIC_0034160/ISIC_0034160_RandomGridShuffle.png


In [21]:
same=0
for i in range(test_genuine_pairs.shape[0]):
    if test_genuine_pairs[i][0].split('_')[3] != test_genuine_pairs[i][1].split('_')[3]:
        same = same + 1

print(same)

0


In [22]:
num_test_genuine_pairs=test_genuine_pairs.shape[0]
print("num test genuine pairs : ", num_test_genuine_pairs)

num test genuine pairs :  8964


In [23]:
test_folderlist[0]

'rasm/siamese/fold1/test_aug/nv/ISIC_0033276'

In [24]:
# Imposter pairs
test_imposter_pairs=[]
for x in test_folderlist:
    current_folder_filenames=np.array(glob(x+"/*"))
    current_folder_filenames_repeated=np.tile(current_folder_filenames,54)  
    np.random.shuffle(current_folder_filenames_repeated)
    current_folder_filenames_repeated=current_folder_filenames_repeated[:54]

    tfl = np.array(test_folderlist)
    other_folderlist = []
    c = x.split('/')[4]    
    for t in tfl:
        cl=t.split('/')[4]     
        if c != cl:
            other_folderlist.append(t)
    
    temp = []
    for z in other_folderlist:
        temp.append(glob(z+"/*"))

    temp=np.array(temp)
    temp=temp.reshape(-1)
    np.random.shuffle(temp)
    temp=temp[:54]
    other_folder_filenames=temp

    test_imposter_pairs.append(np.column_stack((current_folder_filenames_repeated, other_folder_filenames)))

test_imposter_pairs=np.array(test_imposter_pairs)
test_imposter_pairs=test_imposter_pairs.reshape(test_imposter_pairs.shape[0]*test_imposter_pairs.shape[1], test_imposter_pairs.shape[2])
np.random.shuffle(test_imposter_pairs)  

print('test imposter pairs shape : ', test_imposter_pairs.shape)

test imposter pairs shape :  (8964, 2)


In [25]:
print(test_imposter_pairs[0][0])
print(test_imposter_pairs[0][1])

rasm/siamese/fold1/test_aug/nv/ISIC_0027224/ISIC_0027224_JPEGCompression.png
rasm/siamese/fold1/test_aug/mel/ISIC_0027155/ISIC_0027155_ToGray.png


In [26]:
# check same file possibility
same=0
for i in range(test_imposter_pairs.shape[0]):
    if test_imposter_pairs[i][0].split("_")[3] == test_imposter_pairs[i][1].split("_")[3]:
        same=same+1

print('same file duplicated in imposter pairs: ', same)

same file duplicated in imposter pairs:  0


In [27]:
# check same class possibility
same=0
for i in range(test_imposter_pairs.shape[0]):
    if test_imposter_pairs[i][0].split("/")[4] == test_imposter_pairs[i][1].split("/")[4]:
        same=same+1

print('same class duplicated in imposter pairs : ', same)

same class duplicated in imposter pairs :  0


In [28]:
# Test Labels and Save
labels_test_genuine=np.zeros(test_genuine_pairs.shape[0], dtype=np.int8)
labels_test_genuine=labels_test_genuine.reshape(-1,1)
test_genuine_pairs=np.concatenate((test_genuine_pairs, labels_test_genuine), axis=1)

labels_test_imposter=np.ones(test_imposter_pairs.shape[0], dtype=np.int8)
labels_test_imposter=labels_test_imposter.reshape(-1,1)
test_imposter_pairs = np.concatenate((test_imposter_pairs, labels_test_imposter), axis=1)

In [29]:
# save csv files
df=pd.DataFrame(test_genuine_pairs)
df.to_csv(data_root+output_filename_test_genuine, index=False, header=False)  
df=pd.DataFrame(test_imposter_pairs)
df.to_csv(data_root+output_filename_test_imposter, index=False, header=False)